In [ ]:
import pandas as pd

# Load csv
df = pd.read_csv(
    #r"Y:\Huihui\BARseq2\BARseq2_ana\Brain1_1\STARmap_output\Barcodelist.csv"
    r"Y:\Huihui\HH\BARseq2_01202026\Analysis_BARseq2_Transcriptome\Gene_only\Brain1\STARmap_output\gene_mapped.csv"
)

# Marker genes
genes = ["Vglut1", "Gad1", "GFAP", "Cx3cr1","Aldh1l1","no_gene1"]

# Count genes per cell
gene_counts = (
    df[df["Names"].isin(genes)]
    .groupby(["cell_globID", "Names"])
    .size()
    .unstack(fill_value=0)
)

# Ensure all gene columns exist
for g in genes:
    if g not in gene_counts.columns:
        gene_counts[g] = 0

# Classify cells
def classify_cell(row):
    max_gene = row[genes].idxmax()
    max_value = row[genes].max()

    if max_value == 0:
        return "Unknown"

    if (row[genes] == max_value).sum() > 1:
        return "Mixed"

    return {
        "Vglut1": "Vglut1_positive",
        "Gad1": "Gad1_positive",
        "GFAP": "GFAP_positive",
        "Cx3cr1": "Cx3cr1_positive",
        "no_gene1": "no_gene1_positive",
        "Aldh1l1": "Aldh1l1_positive"
    }[max_gene]

gene_counts["cell_type"] = gene_counts.apply(classify_cell, axis=1)

# Prepare outputs
cell_summary = gene_counts.reset_index()

df_with_type = df.merge(
    cell_summary[["cell_globID", "cell_type"]],
    on="cell_globID",
    how="left"
)

# Write to Excel
output_path = r"Y:\Huihui\HH\BARseq2_01202026\Analysis_BARseq2_Transcriptome\Gene_only\Brain1\STARmap_output\cell_gene_classification.xlsx"

with pd.ExcelWriter(output_path, engine="xlsxwriter") as writer:
    cell_summary.to_excel(writer, sheet_name="Cell_Summary", index=False)
    df_with_type.to_excel(writer, sheet_name="Original_With_CellType", index=False)

print(f"Saved Excel file to:\n{output_path}")